# Movement Island Study: Multiple Regression Analysis

**Study:** Psychological Factors Influencing Behavioral Intention to Recommend the Movement Island for Older Adults

**Author:** Aurel Berger

**Date:** February 2026

---

## Overview

This notebook replicates the multiple regression analysis reported in the paper:

- **Table 6:** Multiple Regression Predicting Behavioral Intention
- **Figure 5:** Regression Coefficients with 95% Confidence Intervals

**Model:** BI ~ PU + PEOU + PE + PS

**Important Note:** Sample size (N=16) provides insufficient statistical power for reliable inference. Results demonstrate overall model fit but individual predictors should be interpreted with extreme caution.

## 1. Setup

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

print("✓ Libraries loaded successfully")

## 2. Load Data

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/cleaned_data_full.csv')

# Define item groups for each construct
constructs = {
    'PU': ['PU1', 'PU2', 'PU3'],
    'PEOU': ['PEOU1', 'PEOU2', 'PEOU3'],
    'PE': ['PE1', 'PE2', 'PE3'],
    'PS': ['PS1', 'PS2', 'PS3'],
    'BI': ['BI1', 'BI2', 'BI3']
}

# Calculate composite scores
for construct, items in constructs.items():
    df[f'{construct}_composite'] = df[items].mean(axis=1)

print(f"✓ Data loaded: N = {len(df)}")
print(f"✓ Composite scores calculated")

## 3. Prepare Regression Variables

In [ ]:
# Define predictor and outcome variables
predictors = ['PU_composite', 'PEOU_composite', 'PE_composite', 'PS_composite']
outcome = 'BI_composite'

# Extract data
X = df[predictors]
y = df[outcome]

# Add constant for intercept
X_with_const = sm.add_constant(X)

print("="*70)
print("REGRESSION MODEL SPECIFICATION")
print("="*70)
print(f"\nOutcome Variable:")
print(f"  - Behavioral Intention (BI)")
print(f"\nPredictor Variables:")
print(f"  - Perceived Usefulness (PU)")
print(f"  - Perceived Ease of Use (PEOU)")
print(f"  - Perceived Enjoyment (PE)")
print(f"  - Perceived Safety (PS)")
print(f"\nSample Size: N = {len(df)}")
print(f"Number of Predictors: k = {len(predictors)}")
print(f"Observations per Predictor: {len(df)/len(predictors):.1f}:1")
print(f"\n⚠️  Recommended ratio: ≥ 10-15:1 for stable estimates")
print("="*70)

## 4. Check Multicollinearity

In [ ]:
# Calculate Variance Inflation Factors (VIF)
vif_data = pd.DataFrame()
vif_data['Variable'] = ['PU', 'PEOU', 'PE', 'PS']
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(len(predictors))]

print("="*70)
print("MULTICOLLINEARITY DIAGNOSTICS")
print("="*70)
print(f"\n{'Variable':<15} {'VIF':>15}")
print("-"*70)
for _, row in vif_data.iterrows():
    severity = "⚠️ SEVERE" if row['VIF'] > 10 else "⚠️ High" if row['VIF'] > 5 else "✓ Acceptable"
    print(f"{row['Variable']:<15} {row['VIF']:>15.2f}  {severity}")
print("="*70)
print("\nInterpretation:")
print("  VIF < 5:   Low multicollinearity")
print("  VIF 5-10:  Moderate multicollinearity")
print("  VIF > 10:  Severe multicollinearity (problematic)")
print("\n⚠️  High VIF values indicate strong intercorrelations among predictors,")
print("    making it difficult to isolate unique effects of individual predictors.")
print("="*70)

## 5. Fit Multiple Regression Model

In [ ]:
# Fit OLS regression model
model = sm.OLS(y, X_with_const)
results = model.fit()

# Display full results
print("="*70)
print("MULTIPLE REGRESSION RESULTS")
print("="*70)
print(results.summary())
print("="*70)

## 6. Table 6: Regression Coefficients

In [ ]:
# Extract coefficients and statistics
coef_table = pd.DataFrame({
    'Variable': ['Intercept', 'Perceived Usefulness', 'Perceived Ease of Use', 
                 'Perceived Enjoyment', 'Perceived Safety'],
    'B': results.params.values,
    'SE': results.bse.values,
    't': results.tvalues.values,
    'p': results.pvalues.values,
    'CI_Lower': results.conf_int()[0].values,
    'CI_Upper': results.conf_int()[1].values
})

# Calculate standardized coefficients (beta)
# Standardize predictors and outcome
X_standardized = (X - X.mean()) / X.std()
y_standardized = (y - y.mean()) / y.std()
X_std_with_const = sm.add_constant(X_standardized)
model_std = sm.OLS(y_standardized, X_std_with_const)
results_std = model_std.fit()

# Add beta to table (exclude intercept for beta)
coef_table['Beta'] = [np.nan] + list(results_std.params.values[1:])
coef_table['Beta_CI_Lower'] = [np.nan] + list(results_std.conf_int()[0].values[1:])
coef_table['Beta_CI_Upper'] = [np.nan] + list(results_std.conf_int()[1].values[1:])

# Print formatted table
print("="*90)
print("TABLE 6: MULTIPLE REGRESSION PREDICTING BEHAVIORAL INTENTION (N=16)")
print("="*90)
print(f"\n{'Predictor':<25} {'β':>8} {'95% CI':>20} {'p':>10}")
print("-"*90)
for _, row in coef_table[coef_table['Variable'] != 'Intercept'].iterrows():
    ci_str = f"[{row['Beta_CI_Lower']:.2f}, {row['Beta_CI_Upper']:.2f}]"
    p_str = f"{row['p']:.3f}" if row['p'] >= 0.001 else "<.001"
    print(f"{row['Variable']:<25} {row['Beta']:>8.2f} {ci_str:>20} {p_str:>10}")
print("-"*90)

# Model summary statistics
r_squared = results.rsquared
adj_r_squared = results.rsquared_adj
f_stat = results.fvalue
f_pvalue = results.f_pvalue
cohens_f2 = r_squared / (1 - r_squared)

print(f"\nModel Summary:")
print(f"  R² = {r_squared:.3f}")
print(f"  Adjusted R² = {adj_r_squared:.3f}")
print(f"  F({len(predictors)}, {len(df)-len(predictors)-1}) = {f_stat:.2f}, p = {f_pvalue:.3f}")
print(f"  Cohen's f² = {cohens_f2:.2f} (large effect)")
print("="*90)

# Save coefficient table
coef_table.to_csv('../outputs/table6_regression_coefficients.csv', index=False)
print("\n✓ Table 6 saved to: outputs/table6_regression_coefficients.csv")

## 7. Figure 5: Regression Coefficients with Confidence Intervals

In [ ]:
# Prepare data for plotting (exclude intercept)
plot_data = coef_table[coef_table['Variable'] != 'Intercept'].copy()
plot_data = plot_data.sort_values('Beta', ascending=True)

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Calculate error bars
errors_lower = plot_data['Beta'] - plot_data['Beta_CI_Lower']
errors_upper = plot_data['Beta_CI_Upper'] - plot_data['Beta']

# Create horizontal bar plot
y_pos = np.arange(len(plot_data))
colors_map = ['#0173B2' if b > 0 else '#DE8F05' for b in plot_data['Beta']]

bars = ax.barh(y_pos, plot_data['Beta'], color=colors_map, alpha=0.7, 
               edgecolor='black', linewidth=1)

# Add error bars
ax.errorbar(plot_data['Beta'], y_pos, 
            xerr=[errors_lower, errors_upper],
            fmt='none', ecolor='black', elinewidth=2, capsize=5, capthick=2)

# Zero reference line
ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)

# Labels
ax.set_yticks(y_pos)
ax.set_yticklabels([v.replace('Perceived ', '') for v in plot_data['Variable']])
ax.set_xlabel('Standardized Coefficient (β)', fontweight='bold', fontsize=11)
ax.set_ylabel('Predictor Variable', fontweight='bold', fontsize=11)
ax.set_title('Multiple Regression: Predictors of Behavioral Intention\n(N=16, with 95% CI)', 
             fontweight='bold', pad=15, fontsize=12)

# Grid
ax.grid(True, axis='x', alpha=0.3, linestyle=':')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../figures/figure5_regression_coefficients.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 5 saved to: figures/figure5_regression_coefficients.png")

## 8. Statistical Power Analysis

In [ ]:
# Calculate observed effect size and power
from scipy.stats import f as f_dist

n = len(df)
k = len(predictors)
alpha = 0.05

# Cohen's f² interpretation
print("="*70)
print("EFFECT SIZE AND POWER CONSIDERATIONS")
print("="*70)

print(f"\nObserved Effect Size:")
print(f"  R² = {r_squared:.3f} (explains {r_squared*100:.1f}% of variance)")
print(f"  Cohen's f² = {cohens_f2:.2f}")
print(f"  Interpretation: ", end="")
if cohens_f2 >= 0.35:
    print("Large effect (f² ≥ 0.35)")
elif cohens_f2 >= 0.15:
    print("Medium effect (f² ≥ 0.15)")
elif cohens_f2 >= 0.02:
    print("Small effect (f² ≥ 0.02)")
else:
    print("Negligible effect (f² < 0.02)")

print(f"\nStatistical Power:")
print(f"  Sample size: N = {n}")
print(f"  Predictors: k = {k}")
print(f"  Observations per predictor: {n/k:.1f}:1")
print(f"  Recommended: ≥ 10-15:1 for stable estimates")
print(f"\n⚠️  With N={n} for {k} predictors, statistical power is LOW.")
print(f"    Type II error risk (failing to detect true effects) is HIGH.")
print(f"    Wide confidence intervals reflect this uncertainty.")

print(f"\nRecommended Sample Size:")
print(f"  For medium effect (f² = 0.15), 80% power, α = .05:")
print(f"    N ≥ 85 (for 4 predictors)")
print(f"  For small effect (f² = 0.05):")
print(f"    N ≥ 180 (for 4 predictors)")
print("="*70)

## 9. Summary

In [ ]:
print("="*70)
print("MULTIPLE REGRESSION ANALYSIS SUMMARY")
print("="*70)

print("\n1. Model Fit:")
print(f"   - Overall model: F({len(predictors)}, {len(df)-len(predictors)-1}) = {f_stat:.2f}, p = {f_pvalue:.3f}")
print(f"   - R² = {r_squared:.3f} (explains {r_squared*100:.1f}% of variance)")
print(f"   - Adjusted R² = {adj_r_squared:.3f}")
print(f"   - Effect size: Cohen's f² = {cohens_f2:.2f} (LARGE)")
print(f"   - ✓ Model is statistically significant (p < .05)")

print("\n2. Individual Predictors (Standardized Coefficients):")
for _, row in coef_table[coef_table['Variable'] != 'Intercept'].iterrows():
    sig = '***' if row['p'] < 0.001 else '**' if row['p'] < 0.01 else '*' if row['p'] < 0.05 else 'ns'
    print(f"   - {row['Variable']}: β = {row['Beta']:.3f}, p = {row['p']:.3f} {sig}")

print("\n3. Key Findings:")
strongest = coef_table[coef_table['Variable'] != 'Intercept'].sort_values('Beta', ascending=False).iloc[0]
print(f"   - Strongest predictor: {strongest['Variable']} (β = {strongest['Beta']:.3f})")
print(f"   - No individual predictors reached significance (all p > .10)")
print(f"   - This reflects: (a) severe multicollinearity (VIF > 70)")
print(f"                    (b) insufficient power (N=16, need N≥85+)")

print("\n4. Interpretation:")
print(f"   - Overall model predicts BI well (R² = {r_squared:.3f})")
print(f"   - Cannot isolate unique contributions of individual predictors")
print(f"   - Standardized coefficients suggest tentative ranking:")
for i, (_, row) in enumerate(coef_table[coef_table['Variable'] != 'Intercept'].sort_values('Beta', ascending=False).iterrows(), 1):
    print(f"     {i}. {row['Variable']}: β = {row['Beta']:.3f}")

print("\n" + "="*70)
print("✓ All regression analyses completed successfully!")
print("="*70)

print("\n⚠️  CRITICAL LIMITATION: Results require validation with N≥85-180.")
print("    Current findings demonstrate overall predictive validity but")
print("    cannot reliably identify individual predictor contributions.")